# Testing OT1D
This notebook shows how to use the [OT1D](https://github.com/stegua/ot1d) library and runs a small benchmark that compares **OT1D** with the similar functions provided by [SciPy](https://scipy.org/) and [POT](https://pythonot.github.io/).

The notebook covers:
1. **Basic usage**: computing the Wasserstein distances of order 1 and 2 ($W_1$, $W_2$) with `ot1d.w1` and `ot1d.w2`, and choosing the algorithm with the parameter `method` (new in version 0.5.0).
2. **Optimal transportation plan**: getting the plan, and not only the distance, with `ot1d.w1_plan` and `ot1d.w2_plan`.
3. **SIMD sorting and multithreading** (new in version 0.4.1): AVX-512/AVX2 sorting, and calls from several Python threads.
4. **Correctness check**: comparing both methods of OT1D with SciPy and POT on random instances.
5. **Many samples against one reference** (new in version 0.5.0): `ot1d.reference`, `ot1d.w1_reference` and `ot1d.w1_batch`.
6. **Performance tests**: the same five tests as the script [`OT1D_test.py`](https://github.com/stegua/ot1d/blob/main/python/OT1D_test.py).

### Installation
First, run the following cell to install the required libraries.

In [ ]:
!pip install -q pot "ot1d>=0.5.0"

In [ ]:
import os
from time import perf_counter
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import ot
import scipy
from scipy.stats import wasserstein_distance as W1

import ot1d
from ot1d import OT1D, parasort, simd, set_simd

# Number of threads for the parallel sorting in the performance tests (Colab usually
# provides 2 cores). The default of OT1D is threads=1: see the section on SIMD sorting.
THREADS = os.cpu_count()

try:
    ot1d_version = version('ot1d')
except PackageNotFoundError:
    ot1d_version = '(local build)'

print('ot1d', ot1d_version, '| POT', ot.__version__, '| SciPy', scipy.__version__, '| NumPy', np.__version__)
print('Available cores:', THREADS, '| SIMD sorting:', simd())

### Basic usage
The following cell runs a simple test, as described in this [Kaggle notebook](https://www.kaggle.com/nhan1212/some-statistical-distances). Please refer to that notebook for a detailed explanation of this test.

Since version 0.5.0, the module is called `ot1d` (lowercase), and the main functions are `ot1d.w1(x, y, A=None, B=None, method='simplex', sorting=True, threads=1)` and `ot1d.w2`, with the same parameters: `x` and `y` are the support points of the two measures, and `A` and `B` their (optional) weights. The parameter `threads` is the number of threads used to sort the input: it is the only parallel phase of the algorithm.

The parameter `method` selects one of two exact algorithms, which return the same value (up to rounding):
* `'simplex'` (the default for two samples) is the algorithm of the previous versions: a merge of the two sorted samples, derived from the simplex method for the transportation problem;
* `'ecdf'` integrates the difference between the two quantile functions, using sums precomputed on the first sample. It pays off when one sample is compared with many others: see the section *Many samples against one reference*.

The function `OT1D(x, y, A, B, p=1, ...)` of the previous versions is still available, and it is the same as `w1` or `w2` with `method='simplex'`.

In [ ]:
np.random.seed(13)

N = 1000000

# Uniform samples
x = np.random.uniform(1, 2, N)
y = np.random.uniform(0, 1, N)

print('Wasserstein distance of order 1, W1(x,y) =', ot1d.w1(x, y))
print('Wasserstein distance of order 2, W2(x,y) =', ot1d.w2(x, y))

# The second algorithm, and the function of the previous versions
print("W1(x,y) with method='ecdf'               =", ot1d.w1(x, y, method='ecdf'))
print('W1(x,y) with OT1D(x, y, p=1)             =', OT1D(x, y, p=1))

### Optimal transportation plan
The functions `ot1d.w1_plan` and `ot1d.w2_plan` return the distance and an optimal transportation plan, as a pair `(T, W)`: the row `T[k] = (i, j)` says that the mass `W[k]` moves from `x[i]` to `y[j]`. The plan has at most `m + n - 1` entries.

In the following small example, the first measure has mass 2 in the point 1 and mass 1 in the point 5, while the second measure has mass 1 in each of the points 3, 5, and 10.

In [ ]:
x = np.array([1.0, 5.0])
a = np.array([2.0, 1.0])
y = np.array([5.0, 10.0, 3.0])
b = np.array([1.0, 1.0, 1.0])

z, T, W = ot1d.w2_plan(x, y, a, b)

print('W2 =', z, '\n')
for (i, j), w in zip(T, W):
    print(f'move {w:.1f} from x[{i}] = {x[i]:4.1f} to y[{j}] = {y[j]:4.1f}')

# The plan is feasible: each point sends (receives) exactly its own mass
assert np.allclose(np.bincount(T[:, 0], W, minlength=len(x)), a)
assert np.allclose(np.bincount(T[:, 1], W, minlength=len(y)), b)

### SIMD sorting and multithreading
The key step of the algorithm is sorting the support points. Since version 0.4.1, on x86-64 CPUs OT1D sorts with the vectorized quicksort of [x86-simd-sort](https://github.com/intel/x86-simd-sort) (the same library used by `np.sort`), choosing at runtime the best instruction set supported by the CPU: `'avx512'`, `'avx2'`, or `'none'` (falling back to pdqsort).

The function `simd()` returns the level in use, and `set_simd(level)` selects another one (`set_simd()` restores the best one). The following cell compares the levels supported by this CPU.

In [ ]:
rng = np.random.default_rng(0)
N = 2000000
x, y = rng.normal(0, 1, N), rng.normal(1, 2, N)
a, b = rng.uniform(0.1, 1, N), rng.uniform(0.1, 1, N)
a, b = a / a.sum(), b / b.sum()


def best_time(f, repeat=3):
    t = []
    for _ in range(repeat):
        t0 = perf_counter()
        f()
        t.append(perf_counter() - t0)
    return min(t)


for level in ('avx512', 'avx2', 'none'):
    if set_simd(level) != level:
        print('{:6s}: not supported by this CPU'.format(level))
        continue
    t_u = best_time(lambda: ot1d.w1(x, y))
    t_w = best_time(lambda: ot1d.w1(x, y, a, b))
    print('{:6s}: uniform weights {:.4f} sec | with weights {:.4f} sec'.format(level, t_u, t_w))
print('SIMD sorting in use:', set_simd())

OT1D releases the [GIL](https://docs.python.org/3/glossary.html#term-global-interpreter-lock) while computing, so several distances can be computed in parallel from different Python threads, for instance with a `ThreadPoolExecutor`. In this case, keep the default `threads=1`, to avoid running more threads than cores.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

rng = np.random.default_rng(1)
pairs = [(rng.normal(0, 1, 500000), rng.normal(1, 2, 500000)) for _ in range(16)]

t_seq = best_time(lambda: [ot1d.w1(u, v) for u, v in pairs], repeat=1)
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    t_par = best_time(lambda: list(pool.map(lambda uv: ot1d.w1(*uv), pairs)), repeat=1)

print('16 distances, one after the other: {:.4f} sec'.format(t_seq))
print('16 distances, {} Python threads  : {:.4f} sec (speedup {:.1f})'.format(os.cpu_count(), t_par, t_seq / t_par))

For another example, see [`interpolate.py`](https://github.com/stegua/ot1d/blob/main/python/interpolate.py), which uses the plan to interpolate between two measures.

### Correctness check
Before measuring the runtimes, we check that the three libraries compute the same distances on random instances, with and without weights, and with $m \neq n$. Both methods of OT1D are checked. Note that:
* SciPy only computes $W_1$;
* POT's `ot.lp.wasserstein_1d(..., p=2)` returns $W_2^2$, so we take its square root.

In [ ]:
rng = np.random.default_rng(2021)

worst = 0.0
for trial in range(200):
    m, n = rng.integers(1, 2000, size=2)
    x, y = rng.normal(0, 1, m), rng.normal(1, 2, n)
    a, b = rng.uniform(0.1, 1, m), rng.uniform(0.1, 1, n)
    a, b = a / a.sum(), b / b.sum()

    for w in [(), (a, b)]:  # without and with weights
        ref = {1: W1(x, y, *w), 2: np.sqrt(ot.lp.wasserstein_1d(x, y, *w, p=2))}
        assert np.isclose(ref[1], ot.lp.wasserstein_1d(x, y, *w, p=1))
        for p, f in ((1, ot1d.w1), (2, ot1d.w2)):
            for method in ('simplex', 'ecdf'):
                z = f(x, y, *w, method=method, threads=THREADS)
                err = abs(z - ref[p]) / max(1.0, ref[p])
                worst = max(worst, err)
                assert err < 1e-8, (m, n, p, method, z, ref[p])

print('OT1D (both methods) matches SciPy and POT on 200 random instances: max relative error = {:.1e}'.format(worst))

### Many samples against one reference
Drift monitoring compares many samples (for instance, the chunks of a data stream) with one fixed reference sample, often much larger. For this case, `ot1d.reference(x)` prepares the reference once: it sorts it and stores the prefix sums used by `method='ecdf'`. Then:
* `ot1d.w1_reference(ref, y)` (or `w2_reference`) costs about as much as sorting `y`, whatever the size of the reference;
* `ot1d.w1_batch(ref, ys, threads=...)` (or `w2_batch`) computes the distances to a list of samples, or to the rows of a 2D array, on several threads.

These functions use `method='ecdf'` by default; with `method='simplex'` they merge the sorted reference with every sample, which takes a time proportional to the size of the reference. The following cell compares the two methods on a reference of 10 million points.

In [ ]:
rng = np.random.default_rng(3)
x = rng.lognormal(0, 1, 10_000_000)                          # reference: 10 million points
chunks = [rng.lognormal(0.1, 1, 10_000) for _ in range(100)]  # 100 samples of 10,000 points

t0 = perf_counter()
ref = ot1d.reference(x)
print(f'reference prepared in {perf_counter() - t0:.3f} s, {ref.nbytes / 2**20:.0f} MB')

for method in ['ecdf', 'simplex']:
    t0 = perf_counter()
    z = ot1d.w1_batch(ref, chunks, method=method)
    t = perf_counter() - t0
    print(f'{method:8s}: {1e3 * t / len(chunks):8.3f} ms per sample, first W1 = {z[0]:.12f}')

t0 = perf_counter()
z = ot1d.w1_batch(ref, chunks, threads=os.cpu_count())
print(f'ecdf on {os.cpu_count()} threads: {1e3 * (perf_counter() - t0) / len(chunks):.3f} ms per sample')
print('max difference with a single query:', max(abs(z[i] - ot1d.w1_reference(ref, chunks[i])) for i in range(len(chunks))))

The reference and the samples can have weights (`ot1d.reference(x, A)` and `ot1d.w1_reference(ref, y, B)`), which are normalized to probability measures. With `sorting=False`, a reference is built from data that is already sorted, without sorting it again; a reference can also be pickled, for instance to prepare it once and use it in other processes.

In [ ]:
import pickle

y = chunks[0]

# W2, and weights on the reference and on the sample
a = rng.uniform(0.1, 1, len(x))
b = rng.uniform(0.1, 1, len(y))
ref_w = ot1d.reference(x, a)
print('W2, without weights          :', ot1d.w2_reference(ref, y))
print('W1, weighted reference       :', ot1d.w1_reference(ref_w, y))
print('W1, weighted reference/sample:', ot1d.w1_reference(ref_w, y, b))

# A reference built from sorted data (it is only copied), and a pickled reference
t0 = perf_counter()
ref_s = ot1d.reference(ref.sorted_values, sorting=False)
print(f'reference from sorted data prepared in {perf_counter() - t0:.3f} s')
ref_p = pickle.loads(pickle.dumps(ref))
assert np.isclose(ot1d.w1_reference(ref_s, y), z[0], rtol=1e-12)
assert np.isclose(ot1d.w1_reference(ref_p, y), z[0], rtol=1e-12)

# The functions on two samples return the same value, with either method
print("ot1d.w1(x, y)                :", ot1d.w1(x, y))
print("ot1d.w1(x, y, method='ecdf') :", ot1d.w1(x, y, method='ecdf'))
print("ot1d.w1_reference(ref, y)    :", ot1d.w1_reference(ref, y))

### Performance tests
The following cells define and run five tests that assess and compare the performance of **OT1D**:

* **TEST 1**: unsorted input, $n = m = 5\,000\,000$;
* **TEST 2**: the same input, sorted once with `parasort`, and then solved with `sorting=False`;
* **TEST 3**: average runtime over 30 random instances with unsorted input, $n = m = 500\,000$;
* **TEST 4**: as TEST 3, with input sorted in advance;
* **TEST 5**: $W_2$ computed by OT1D against POT, on instances from 1 to $5\,000\,000$ points.

The tests call the function `OT1D` of the previous versions, as the script does: it is the same as `ot1d.w1` and `ot1d.w2` with `method='simplex'`.

In TESTS 3 and 4, the speedup is the ratio between the runtime of a library and the runtime of OT1D. Running all the tests might take a few minutes.

**REMARK**: the tests use `threads=THREADS` (all the available cores) for the parallel sorting. With SIMD sorting, a single thread is already fast, and the extra threads are used only for samples without weights of at least 5 million points (from 4 threads); Colab usually provides only 2 cores, so their benefit is limited there.

In [ ]:
def timed(f, *args, **kwargs):
    # Return the value of f(*args, **kwargs) and its runtime in seconds
    t0 = perf_counter()
    z = f(*args, **kwargs)
    return z, perf_counter() - t0


def solvers(sorting, threads):
    # The solvers to compare, as functions of (x, y, weights, p).
    # Note: POT returns W2^2, so we take the square root.
    return {
        'SciPy': lambda x, y, w, p: W1(x, y, *w) if p == 1 else None,
        'POT':   lambda x, y, w, p: ot.lp.wasserstein_1d(x, y, *w, p=p) ** (1 / p),
        'OT1D':  lambda x, y, w, p: OT1D(x, y, *w, p=p, sorting=sorting, threads=threads),
    }


CASES = [(1, False), (2, False), (1, True), (2, True)]


def case_name(p, weighted):
    return 'Testing W{}, samples {}'.format(p, 'with weights' if weighted else 'of deltas, n=m')


def single_run(x, y, a, b, sorting, threads):
    for p, weighted in CASES:
        print('\n' + case_name(p, weighted))
        w = (a, b) if weighted else ()
        for name, f in solvers(sorting, threads).items():
            if name == 'SciPy' and p == 2:
                continue  # SciPy only computes W1
            z, t = timed(f, x, y, w, p)
            print('{:5s}: z = {:.8f}  time = {:.4f}'.format(name, z, t))


def Test1(x, y, a, b, threads):
    print('\n--------------- TEST 1: Unsorted input --------------------')
    print('For OT1D using', threads, 'threads')
    single_run(x, y, a, b, sorting=True, threads=threads)


def Test2(x, y, a, b, threads):
    print('\n--------------- TEST 2: Sorted input --------------------')
    print('For OT1D using', threads, 'threads\n')
    # Sort copies of the input data (with their weights) once for all
    x, y, a, b = x.copy(), y.copy(), a.copy(), b.copy()
    t0 = perf_counter()
    parasort(x, a, threads)
    parasort(y, b, threads)
    print('Parallel sorting: time =', round(perf_counter() - t0, 3), 'sec')
    single_run(x, y, a, b, sorting=False, threads=threads)


def average_runtimes(title, presorted, N=500000, samples=30, threads=1):
    print('\n--------------- {} --------------------'.format(title))
    print('For OT1D using', threads, 'threads\n')
    rng = np.random.default_rng(13)
    times = {}
    t_sort = 0.0
    print('running test', end=' ')
    for _ in range(samples):
        print('.', end=' ', flush=True)
        x, y = rng.uniform(1, 10, N), rng.uniform(3, 5, N)
        a, b = rng.uniform(500, 1000, N), rng.uniform(0, 1000, N)
        a, b = a / a.sum(), b / b.sum()
        if presorted:
            t0 = perf_counter()
            parasort(x, a, threads)
            parasort(y, b, threads)
            t_sort += perf_counter() - t0
        for p, weighted in CASES:
            w = (a, b) if weighted else ()
            for name, f in solvers(sorting=not presorted, threads=threads).items():
                if name == 'SciPy' and p == 2:
                    continue
                _, t = timed(f, x, y, w, p)
                times[name, p, weighted] = times.get((name, p, weighted), 0.0) + t
    print()
    if presorted:
        print('Parallel sorting: time =', round(t_sort / samples, 3), 'sec')
    for p, weighted in CASES:
        print('\n' + case_name(p, weighted))
        for name in ('SciPy', 'POT', 'OT1D'):
            if (name, p, weighted) in times:
                t = times[name, p, weighted]
                print('{:5s}: average time = {:.4f}  speedup = {:.1f}'.format(
                    name, t / samples, t / times['OT1D', p, weighted]))


def Test3(threads, **kwargs):
    average_runtimes('TEST 3: Unsorted input (average runtime)', presorted=False, threads=threads, **kwargs)


def Test4(threads, **kwargs):
    average_runtimes('TEST 4: Sorted input (average runtime)', presorted=True, threads=threads, **kwargs)


def Test5(threads, seed=13):
    print('\n--------------- TEST 5: W2 against POT (correctness) --------------------')
    print('Comparing OT1D with sqrt(POT), since POT returns W2^2\n')
    rng = np.random.default_rng(seed)
    worst = 0.0
    for m, n in [(1, 1), (2, 3), (10, 10), (17, 5), (1000, 1000), (1000, 777),
                 (100000, 100000), (100000, 65537), (5000000, 5000000)]:
        x, y = rng.normal(0, 1, m), rng.normal(3, 2, n)
        a, b = rng.uniform(0.1, 1, m), rng.uniform(0.1, 1, n)
        a, b = a / a.sum(), b / b.sum()
        for weighted in (False, True):
            w = (a, b) if weighted else ()
            ref, t_pot = timed(lambda: np.sqrt(ot.lp.wasserstein_1d(x, y, *w, p=2)))
            z, t_sg = timed(OT1D, x, y, *w, p=2, threads=threads)
            err = abs(z - ref) / max(1.0, abs(ref))
            worst = max(worst, err)
            assert err < 1e-8, (m, n, weighted, z, ref)
            print('m = {:>7}, n = {:>7}, weights = {:<5} | POT: z = {:.8f} time = {:.4f} | '
                  'OT1D: z = {:.8f} time = {:.4f}'.format(m, n, str(weighted), ref, t_pot, z, t_sg))
    print('\nAll W2 values match POT: max relative error = {:.2e}'.format(worst))

In [ ]:
# Example 1 from: https://www.kaggle.com/nhan1212/some-statistical-distances
rng = np.random.default_rng(13)
N = 5000000

x = rng.normal(0, 1, N)
y = rng.normal(10, 1, N)

# Random weights for samples
a = rng.uniform(500, 1000, N)
b = rng.uniform(0, 1000, N)
a, b = a / a.sum(), b / b.sum()

Test1(x, y, a, b, threads=THREADS)
Test2(x, y, a, b, threads=THREADS)

In [ ]:
Test3(threads=THREADS)
Test4(threads=THREADS)

In [ ]:
Test5(threads=THREADS)

### Related projects
* [dotlib](https://github.com/stegua/dotlib): OT1D is part of this larger project, which develops Optimal Transport algorithms based on efficient Linear Programming implementations.
* [not1d](https://github.com/pietroppeter/not1d) by Pietro Peterlongo: a partial port of OT1D to [Nim](https://nim-lang.org/), which can also be used from Python.

### Author and maintainer
Please contact us by email if you encounter any issues.

Stefano Gualandi, stefano.gualandi@gmail.com.